# Packages and Data Instantiation

In [1]:
from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import ndtr
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MultiLabelBinarizer

def l_dir(d_p: str = "../data") -> Dict[str, pd.DataFrame]:
    """Loads all CSV files from a directory efficiently into memory.

    Args:
        d_p: Directory path containing the target files.

    Returns:
        Dictionary mapping filename stems to loaded DataFrames.
    """
    return {f.stem: pd.read_csv(f, engine="pyarrow") for f in Path(d_p).glob("*.csv")}

def l_xl(d_p: str = "../data") -> Dict[str, pd.DataFrame]:
    """Loads all Excel files from a directory into memory.

    Args:
        d_p: Directory path containing the target .xlsx files.

    Returns:
        Dictionary mapping filename stems to loaded DataFrames.
    """
    return {
        f.stem: pd.read_excel(f, engine="openpyxl", index_col=0, parse_dates=True)
        for f in Path(d_p).glob("*.xlsx")
    }

In [2]:
d_m = l_dir()

df_crs = d_m.get("crsp_daily_prices")
df_evt = d_m.get("capitaliq_key_developments")
df_ff = d_m.get("fama_french_5f_daily")
df_ibs = d_m.get("ibes_eps_summary")
df_main = d_m.get("integrated_feature_matrix")

In [3]:
class EDA:
    """Exploratory Data Analysis diagnostics for time-series feature matrices."""

    def __init__(self, m: pd.DataFrame):
        if not isinstance(m, pd.DataFrame):
            raise ValueError("The input 'm' must be a pandas DataFrame. The variable may have been overwritten or failed to load.")
        self.m = m
        self.c = m.columns

    def s_chk(self) -> pd.DataFrame:
        """Calculates column sparsity and missingness."""
        n = self.m.isnull().sum()
        p = (n / len(self.m)) * 100
        t = self.m.dtypes
        return pd.DataFrame({'n_ms': n, 'p_ms': p, 'd_ty': t}).sort_values('p_ms', ascending=False)

    def t_chk(self, d_c: str) -> Dict[str, str]:
        """Validates temporal continuity and identifies boundary limits."""
        dt = pd.to_datetime(self.m[d_c])
        return {
            'strt': str(dt.min().date()),
            'end': str(dt.max().date()),
            'n_dys': str(dt.nunique()),
            'gaps': str(len(dt) - dt.nunique())
        }

    def tgt_sts(self, t_c: str) -> pd.DataFrame:
        """Computes distribution statistics for the target variable."""
        if t_c not in self.c:
            return pd.DataFrame()
        return self.m[[t_c]].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

In [4]:
eda = EDA(df_main)
eda.s_chk()

,n_ms,p_ms,d_ty
keydeveventtypeid,750,83.426029,object
headline,750,83.426029,object
dlyret,1,0.111235,float64
dlycaldt,0,0.000000,object
at,0,0.000000,float64
lt,0,0.000000,float64
dlyvol,0,0.000000,float64
curcd,0,0.000000,object
meanest_fy1,0,0.000000,float64
meanest_fy2,0,0.000000,float64


In [5]:
eda.t_chk('dlycaldt')

{'strt': '2022-06-02', 'end': '2025-12-31', 'n_dys': '899', 'gaps': '0'}

In [6]:
eda.tgt_sts('dlyret')

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
dlyret,898.0,0.000071,0.019793,-0.146109,-0.052006,-0.031786,-0.009884,-0.000206,0.012205,0.029974,0.049173,0.087826


# Econometrics

In [7]:
# import numpy as np
# import pandas as pd
# from dask import compute, delayed
# import statsmodels.api as sm
# from statsmodels.tsa.stattools import adfuller, kpss, bds
# from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox, het_white, het_breuschpagan
# from statsmodels.tsa.ar_model import ar_select_order, AutoReg
# from statsmodels.stats.stattools import durbin_watson, jarque_bera
# import warnings
# from statsmodels.tools.sm_exceptions import InterpolationWarning

# class TSDiagnostics:
#     def __init__(self, y: np.ndarray):
#         self.y = np.ascontiguousarray(np.asarray(y, dtype=np.float64).ravel())
#         self.y = self.y[np.isfinite(self.y)]
#         self.dy = np.diff(self.y)
#         self.n = self.y.shape[0]
#         self.dy_ok = self.dy.size > 0 and np.ptp(self.dy) > 0.0

#     @staticmethod
#     def _run_test(func, out_dict, key_map, *args, **kwargs):
#         try:
#             res = func(*args, **kwargs)
#             if res is not None:
#                 for key, idx in key_map.items():
#                     out_dict[key] = res[idx] if isinstance(res, tuple) else res
#         except Exception:
#             pass

#     @delayed
#     def _test_stationarity(self, max_lag: int) -> dict:
#         out = {}
#         self._run_test(adfuller, out, {'adf_stat': 0, 'adf_pval': 1}, self.y, maxlag=max_lag, autolag='AIC')
#         with warnings.catch_warnings():
#             warnings.simplefilter("ignore", InterpolationWarning)
#             self._run_test(kpss, out, {'kpss_stat': 0, 'kpss_pval': 1}, self.y, regression='c', nlags='auto')

#         if self.dy_ok:
#             self._run_test(adfuller, out, {'adf_d_stat': 0, 'adf_d_pval': 1}, self.dy, maxlag=max_lag, autolag='AIC')
#             with warnings.catch_warnings():
#                 warnings.simplefilter("ignore", InterpolationWarning)
#                 self._run_test(kpss, out, {'kpss_d_stat': 0, 'kpss_d_pval': 1}, self.dy, regression='c', nlags='auto')
#         return out

#     @delayed
#     def _test_serial_corr(self, lags: int) -> dict:
#         out = {}
#         try:
#             out['lb_pval'] = acorr_ljungbox(self.y, lags=[lags], return_df=False).iloc[0, 1]
#         except Exception:
#             pass

#         if self.dy_ok:
#             try:
#                 out['lb_d_pval'] = acorr_ljungbox(self.dy, lags=[lags], return_df=False).iloc[0, 1]
#                 out['lb_abs_d_pval'] = acorr_ljungbox(np.abs(self.dy), lags=[lags], return_df=False).iloc[0, 1]
#                 out['lb_d2_pval'] = acorr_ljungbox(self.dy**2, lags=[lags], return_df=False).iloc[0, 1]
#             except Exception:
#                 pass
#             self._run_test(het_arch, out, {'arch_lm_pval': 1}, self.dy, nlags=lags)
#         return out

#     @delayed
#     def _test_residuals(self) -> dict:
#         out = {}
#         xc = np.ones((self.n, 1), dtype=np.float64)
#         xh = np.column_stack((np.ones(self.n, dtype=np.float64), np.arange(self.n, dtype=np.float64)))

#         try:
#             ols = sm.OLS(self.y, xc).fit()
#             r = np.asarray(ols.resid, dtype=np.float64)

#             try:
#                 out["dw_stat"] = float(durbin_watson(r))
#             except Exception:
#                 pass

#             self._run_test(het_white, out, {"hetw_lm_p": 1}, r, xh)
#             self._run_test(het_breuschpagan, out, {"hetbp_lm_p": 1}, r, xh, robust=True)
#             self._run_test(jarque_bera, out, {"jb_p": 1, "jb_skew": 2, "jb_kurt": 3}, r)

#         except Exception:
#             pass

#         if self.dy_ok:
#             try:
#                 s, p = bds(self.dy, max_dim=2)
#                 out.update({"bds_stat": s[0] if isinstance(s, np.ndarray) else s,
#                             "bds_p": p[0] if isinstance(p, np.ndarray) else p})
#             except Exception:
#                 pass
#         return out

#     @delayed
#     def _fit_ar(self, max_lag: int) -> dict:
#         out = {}
#         try:
#             sel = ar_select_order(self.y, maxlag=max_lag, ic='bic', trend='c')
#             lags = sel.ar_lags if (sel.ar_lags is not None and len(sel.ar_lags) > 0) else [1]
#             mod = AutoReg(self.y, lags=lags, trend='c').fit()
#             out.update({'ar_lags': len(lags), 'ar_bic': mod.bic})
#         except Exception:
#             pass

#         if self.dy_ok:
#              try:
#                 sel_d = ar_select_order(self.dy, maxlag=max_lag, ic='bic', trend='c')
#                 lags_d = sel_d.ar_lags if (sel_d.ar_lags is not None and len(sel_d.ar_lags) > 0) else [1]
#                 mod_d = AutoReg(self.dy, lags=lags_d, trend='c').fit()
#                 out.update({'ar_d_lags': len(lags_d), 'ar_d_bic': mod_d.bic})
#              except Exception:
#                 pass
#         return out

#     @classmethod
#     def profile_assets(cls, df: pd.DataFrame, target_col: str, asset_col: str, min_obs: int = 50) -> pd.DataFrame:
#         if asset_col not in df.columns:
#             df = df.copy()
#             df[asset_col] = 'TARGET'

#         tasks = {}
#         for asset, group in df.groupby(asset_col, sort=False):
#             series = group[target_col].to_numpy()
#             if np.isfinite(series).sum() >= min_obs:
#                 engine = cls(series)
#                 dyn_lag = max(1, min(12, engine.n // 10))

#                 t1 = engine._test_stationarity(dyn_lag)
#                 t2 = engine._test_serial_corr(dyn_lag)
#                 t3 = engine._test_residuals()
#                 t4 = engine._fit_ar(dyn_lag)

#                 tasks[asset] = delayed(lambda *dicts: {k: v for d in dicts for k, v in d.items()})(t1, t2, t3, t4)

#         results = compute(tasks)[0]
#         return pd.DataFrame.from_dict(results, orient='index')

In [8]:
# df_econometrics = TSDiagnostics.profile_assets(df_main, target_col='dlyret', asset_col='ticker')
# print(df_econometrics)

# fPCA

In [9]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd


class DataLoader:
    """Parses the combined forwards + options workbook into clean frames.

    Attributes:
        path: Workbook location.
        asof: Snapshot date used to compute option time-to-expiry.
        iv_floor: Lower plausibility clip on implied volatility.
        iv_cap: Upper plausibility clip on implied volatility.
    """

    _TICKER_RE = re.compile(
        r"^\s*(?P<root>\S+)\s+\S+\s+"
        r"(?P<expiry>\d{2}/\d{2}/\d{2})\s+"
        r"(?P<type>[CP])(?P<strike>\d+(?:\.\d+)?)"
    )

    def __init__(
        self,
        path: str | Path,
        asof: str | pd.Timestamp = "2026-04-07",
        iv_floor: float = 0.02,
        iv_cap: float = 3.0,
    ):
        self.path = Path(path)
        if not self.path.exists():
            raise FileNotFoundError(f"Workbook not found: {self.path}")
        self.asof = pd.Timestamp(asof)
        self.iv_floor = iv_floor
        self.iv_cap = iv_cap

    def load(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Parses all three target sheets in a single workbook open.

        Returns:
            Tuple of (brent_curve, jkm_curve, options_chain).
        """
        with pd.ExcelFile(self.path, engine="openpyxl") as xl:
            brent = self._curve(xl, "Sheet6", "CO")
            jkm = self._curve(xl, "Sheet7", "JKL")
            opt = self._options(xl, "Sheet8")
        return brent, jkm, opt

    def _curve(self, xl: pd.ExcelFile, sheet: str, root: str) -> pd.DataFrame:
        """Parses a Bloomberg forward curve sheet into a wide tenor matrix.

        Args:
            xl: Open ExcelFile handle.
            sheet: Sheet name to parse.
            root: Contract root (e.g. "CO", "JKL").

        Returns:
            DataFrame indexed by business date with integer tenor columns.
        """
        df = xl.parse(sheet, header=0)
        df = df.rename(columns={df.columns[0]: "date"})
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"]).set_index("date").sort_index()

        tenor_re = re.compile(rf"^{root}(\d+)\s+(?:COMB\s+)?Comdty$", re.IGNORECASE)
        rename = {c: int(m.group(1)) for c in df.columns if (m := tenor_re.match(str(c)))}

        df = df[list(rename)].rename(columns=rename)
        df = df.reindex(columns=sorted(df.columns)).apply(pd.to_numeric, errors="coerce")
        return df.dropna(how="all").dropna(axis=1, how="all")

    def _options(self, xl: pd.ExcelFile, sheet: str) -> pd.DataFrame:
        """Parses the WDS options chain into a long-format frame.

        Args:
            xl: Open ExcelFile handle.
            sheet: Sheet name to parse.

        Returns:
            DataFrame with one row per valid option quote.
        """
        df = xl.parse(
            sheet,
            header=None,
            skiprows=1,
            names=["ticker", "strike_bbg", "type_bbg", "px_last", "px_settle", "ivol"],
        )
        df = df.dropna(subset=["ticker"]).reset_index(drop=True)

        parsed = df["ticker"].astype(str).str.extract(self._TICKER_RE)
        df["root"] = parsed["root"]
        df["expiry"] = pd.to_datetime(parsed["expiry"], format="%m/%d/%y", errors="coerce")
        df["type"] = parsed["type"].map({"C": "Call", "P": "Put"})
        df["strike"] = pd.to_numeric(parsed["strike"], errors="coerce")
        df = df.dropna(subset=["expiry", "type", "strike"]).reset_index(drop=True)

        for c in ("px_last", "px_settle", "ivol"):
            df[c] = pd.to_numeric(df[c], errors="coerce")

        df["price"] = df["px_settle"].where(df["px_settle"].notna(), df["px_last"])
        df["iv"] = (df["ivol"] / 100.0).clip(lower=self.iv_floor, upper=self.iv_cap)
        df.loc[df["ivol"].isna(), "iv"] = np.nan
        df["T"] = (df["expiry"] - self.asof).dt.days / 365.25

        df = df[(df["T"] > 0) & (df["price"] > 0) & (df["strike"] > 0)].dropna(subset=["price", "strike"])
        df = (
            df.sort_values(["expiry", "type", "strike"])
            .drop_duplicates(subset=["expiry", "type", "strike"], keep="last")
            .reset_index(drop=True)
        )
        return df[["root", "expiry", "T", "type", "strike", "price", "iv", "px_last", "px_settle", "ivol", "ticker"]]

In [10]:
candidates = [
    Path("Sus_Data.xlsx"),
    Path("data/Sus_Data.xlsx"),
    Path("../data/Sus_Data.xlsx"),
]
workbook = next((p for p in candidates if p.exists()), None)
if workbook is None:
    raise FileNotFoundError("Workbook not found in expected locations.")

loader = DataLoader(workbook, asof="2026-04-07")
df_brent, df_jkm, df_opt = loader.load()

In [11]:
df_brent

,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,51.09,51.16,51.16,51.08,50.94,50.80,50.66,50.50,50.36,50.23,...,49.78,49.72,49.69,49.64,49.58,49.51,49.44,49.37,49.30,49.27
2021-01-05,53.60,53.54,53.43,53.24,53.00,52.77,52.54,52.31,52.09,51.87,...,51.09,50.97,50.89,50.80,50.69,50.58,50.47,50.35,50.24,50.17
2021-01-06,54.30,54.18,54.00,53.75,53.45,53.16,52.86,52.56,52.28,52.02,...,51.04,50.89,50.77,50.65,50.52,50.38,50.24,50.12,49.99,49.90
2021-01-07,54.38,54.30,54.13,53.89,53.59,53.30,53.01,52.72,52.44,52.20,...,51.30,51.17,51.06,50.95,50.83,50.70,50.57,50.45,50.33,50.24
2021-01-08,55.99,55.83,55.58,55.27,54.91,54.57,54.24,53.92,53.61,53.33,...,52.27,52.10,51.97,51.83,51.69,51.54,51.39,51.24,51.10,51.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-02,109.03,99.44,91.43,85.95,82.54,80.47,79.02,77.79,76.80,76.06,...,74.06,73.82,73.58,73.37,73.15,72.90,72.67,72.52,72.39,72.29
2026-04-06,109.77,100.12,92.24,86.96,83.73,81.83,80.49,79.30,78.32,77.55,...,75.30,75.03,74.77,74.54,74.30,74.02,73.77,73.60,73.46,73.33
2026-04-07,109.27,100.06,92.77,87.95,84.94,83.13,81.80,80.61,79.60,78.79,...,76.34,76.05,75.78,75.53,75.27,74.98,74.72,74.53,74.37,74.22


In [12]:
df_jkm

,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,14.930,10.325,7.19,6.565,6.215,6.240,6.325,6.490,6.675,7.095,...,6.085,5.820,5.795,5.755,5.860,5.965,5.990,6.355,6.955,6.855
2021-01-05,15.105,9.800,6.85,6.190,6.040,6.075,6.150,6.275,6.550,6.970,...,6.005,5.750,5.720,5.605,5.710,5.810,5.835,6.195,6.780,6.825
2021-01-06,15.550,9.550,6.50,6.065,5.950,5.985,6.065,6.215,6.500,6.925,...,5.980,5.725,5.695,5.700,5.805,5.905,5.930,6.295,6.890,6.810
2021-01-07,17.250,10.775,7.00,6.500,6.240,6.265,6.320,6.465,6.725,7.140,...,6.005,5.750,5.720,5.735,5.840,5.945,5.970,6.335,6.935,6.865
2021-01-08,17.250,12.000,7.29,6.615,6.415,6.440,6.500,6.650,6.900,7.240,...,6.055,5.795,5.770,5.795,5.900,6.005,6.030,6.400,7.005,6.940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-02,19.965,18.340,18.61,18.405,18.205,17.545,17.450,17.770,17.515,17.350,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-03,19.965,18.340,18.61,18.405,18.205,17.545,17.450,17.770,17.515,17.350,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-06,19.965,18.340,18.61,18.405,18.205,17.545,17.450,17.770,17.515,17.350,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
class EmpiricalFPCA:
    """Functional PCA for discretely-sampled forward curves.

    Fits a covariance operator on PCHIP-smoothed curves and extracts the
    leading eigenfunctions via a symmetric eigendecomposition.

    Attributes:
        n_components: Number of principal components to retain.
        grid: Uniform tenor grid produced during fit.
        mean: Empirical mean curve on the grid.
        eigfn: Matrix of eigenfunctions (rows) on the grid.
        eigval: Eigenvalues in descending order.
        evr: Explained variance ratio per retained component.
    """

    def __init__(self, n_components: int = 3):
        if n_components < 1:
            raise ValueError("n_components must be >= 1.")
        self.n_components = n_components
        self.grid: np.ndarray | None = None
        self.mean: np.ndarray | None = None
        self.eigfn: np.ndarray | None = None
        self.eigval: np.ndarray | None = None
        self.evr: np.ndarray | None = None
        self._fit = False

    def _smooth(self, X: pd.DataFrame) -> pd.DataFrame:
        """Projects raw tenor observations onto a uniform integer grid.

        Args:
            X: Raw forward curves with numeric maturity columns.

        Returns:
            DataFrame of PCHIP-interpolated curves on the uniform grid.

        Raises:
            ValueError: If the input is empty or has non-numeric columns.
        """
        if X.empty:
            raise ValueError("Input matrix is empty.")
        try:
            t = np.asarray(X.columns, dtype=float)
        except ValueError as exc:
            raise ValueError("Columns must be numeric maturities.") from exc

        grid = np.arange(int(np.floor(t.min())), int(np.ceil(t.max())) + 1, dtype=float)
        wide = X.copy()
        wide.columns = t
        wide = wide.reindex(columns=np.union1d(t, grid))
        return wide.interpolate(method="pchip", axis=1, limit_direction="both")[grid]

    def fit(self, X: pd.DataFrame) -> "EmpiricalFPCA":
        """Fits the empirical covariance operator and extracts eigenfunctions.

        Args:
            X: Historical training surface with numeric tenor columns.

        Returns:
            Fitted instance.

        Raises:
            ValueError: If n_components exceeds the grid dimension.
            RuntimeError: If no valid curves remain after alignment.
        """
        smooth = self._smooth(X).dropna(how="any", axis=0)
        if smooth.empty:
            raise RuntimeError("Zero valid curves remain after smoothing.")

        Y = smooth.to_numpy()
        n, d = Y.shape
        if self.n_components > d:
            raise ValueError("n_components exceeds grid dimension.")

        self.grid = np.asarray(smooth.columns, dtype=float)
        self.mean = Y.mean(axis=0)
        centered = Y - self.mean
        cov = centered.T @ centered / (n - 1)

        val, vec = np.linalg.eigh(cov)
        self.eigval = val[-self.n_components:][::-1]
        self.eigfn = vec[:, -self.n_components:][:, ::-1].T

        trace = np.trace(cov)
        self.evr = self.eigval / trace if trace > 0 else np.zeros(self.n_components)
        self._fit = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Projects curves onto the fitted eigenfunctions.

        Args:
            X: Curves to decompose.

        Returns:
            DataFrame of principal component scores.

        Raises:
            RuntimeError: If the model has not been fitted.
        """
        if not self._fit:
            raise RuntimeError("Model must be fitted before transform.")
        smooth = self._smooth(X)
        scores = (smooth.to_numpy() - self.mean) @ self.eigfn.T
        return pd.DataFrame(
            scores,
            index=smooth.index,
            columns=[f"PC{i+1}" for i in range(self.n_components)],
        )

In [14]:
fpca_brent = EmpiricalFPCA(n_components=3).fit(df_brent)
fpca_jkm   = EmpiricalFPCA(n_components=3).fit(df_jkm)

In [15]:
fpca_brent.eigval, fpca_brent.evr
fpca_jkm.eigval, fpca_jkm.evr

(array([391511.29812232,   2705.94704726,   1371.5031564 ]),
 array([0.98912401, 0.00683637, 0.003465  ]))

In [16]:
brent_scores = fpca_brent.transform(df_brent)
jkm_scores = fpca_jkm.transform(df_jkm)

In [17]:
brent_scores

,PC1,PC2,PC3
date,,,
2021-01-04,-118.844004,-10.775542,3.102677
2021-01-05,-110.955401,-11.700128,3.421948
2021-01-06,-110.178557,-13.005062,3.770882
2021-01-07,-109.161895,-12.470533,3.561335
2021-01-08,-103.722556,-12.710558,3.622724
...,...,...,...
2026-04-02,25.328602,-26.619283,-14.563678
2026-04-06,31.005664,-25.114976,-13.969002
2026-04-07,35.422878,-23.033612,-12.664179


In [18]:
jkm_scores

,PC1,PC2,PC3
date,,,
2021-01-04,112.139670,-20.033520,40.241481
2021-01-05,111.868479,-20.271756,40.939924
2021-01-06,112.004531,-20.077689,41.202467
2021-01-07,112.120952,-20.157341,39.528542
2021-01-08,112.278998,-20.131392,38.688474
...,...,...,...
2026-04-02,1512.783054,121.274000,-1.776856
2026-04-03,1512.783054,121.274000,-1.776856
2026-04-06,1512.783054,121.274000,-1.776856


In [19]:
print(fpca_brent.eigfn.shape)
print(fpca_brent.eigfn)

print(fpca_jkm.eigfn.shape)
print(fpca_jkm.eigfn)

(3, 24)
[[ 0.27734017  0.26574768  0.25513935  0.24561449  0.2371162   0.22949063
   0.22269702  0.21652808  0.21088102  0.20578563  0.20118328  0.19691726
   0.19284205  0.18873814  0.18468351  0.18074071  0.17686831  0.17308308
   0.16934446  0.16565771  0.1620648   0.15858825  0.15529435  0.15206213]
 [-0.51066672 -0.39173254 -0.28682863 -0.20052671 -0.13227403 -0.07778698
  -0.03446097  0.00176248  0.03271696  0.05914616  0.08062266  0.09891047
   0.11460089  0.12958528  0.14385855  0.15676314  0.16850321  0.17972591
   0.19070086  0.20199156  0.2127032   0.22240921  0.23061523  0.23820984]
 [-0.51310881 -0.19623134  0.01620898  0.14947892  0.21761957  0.24350734
   0.24277751  0.2298718   0.21181416  0.18878394  0.1603465   0.12993382
   0.09681458  0.06096366  0.02526869 -0.01189248 -0.04946495 -0.0866603
  -0.12492189 -0.16242342 -0.20013785 -0.23790142 -0.2742791  -0.31044641]]
(3, 24)
[[ 0.00268935  0.0026914   0.00283981  0.00288661  0.00286482  0.0027757
   0.00272102  0.002

In [20]:
display(pd.DataFrame(fpca_brent.eigfn, columns=fpca_brent.grid, index=["PC1","PC2","PC3"]))
display(pd.DataFrame(fpca_jkm.eigfn, columns=fpca_jkm.grid, index=["PC1","PC2","PC3"]))

,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,...,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0
PC1,0.277340,0.265748,0.255139,0.245614,0.237116,0.229491,0.222697,0.216528,0.210881,0.205786,...,0.184684,0.180741,0.176868,0.173083,0.169344,0.165658,0.162065,0.158588,0.155294,0.152062
PC2,-0.510667,-0.391733,-0.286829,-0.200527,-0.132274,-0.077787,-0.034461,0.001762,0.032717,0.059146,...,0.143859,0.156763,0.168503,0.179726,0.190701,0.201992,0.212703,0.222409,0.230615,0.238210
PC3,-0.513109,-0.196231,0.016209,0.149479,0.217620,0.243507,0.242778,0.229872,0.211814,0.188784,...,0.025269,-0.011892,-0.049465,-0.086660,-0.124922,-0.162423,-0.200138,-0.237901,-0.274279,-0.310446


,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,...,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0
PC1,0.002689,0.002691,0.002840,0.002887,0.002865,0.002776,0.002721,0.002688,0.002594,0.002500,...,0.004229,0.012672,0.029814,0.059200,0.104406,0.169120,0.256997,0.371703,0.516859,0.696050
PC2,-0.020267,-0.021292,-0.022095,-0.022494,-0.022311,-0.021110,-0.020367,-0.019711,-0.019515,-0.019461,...,0.027827,0.104968,0.207469,0.315578,0.405379,0.447639,0.412005,0.265776,-0.022583,-0.483661
PC3,-0.263769,-0.271457,-0.282405,-0.283865,-0.277243,-0.266240,-0.260525,-0.257036,-0.255029,-0.254114,...,-0.180639,-0.102721,-0.066247,-0.056293,-0.043640,-0.026238,0.001701,0.019703,0.023055,0.007902


# Causal Estimation

In [21]:
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional
from scipy.stats import norm
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.preprocessing import MultiLabelBinarizer

class FeatureMatrix:
    """Engineers EWMA confounders and forward cumulative abnormal returns.

    Attributes:
        horizon: Trading days for the forward CAR target.
        span: EWMA span for volatility and momentum confounders.
    """

    _BASE = ("mktrf", "smb", "hml", "rmw", "cma", "rf", "vol_ewm", "mom_ewm")
    _OPT = ("meanest_fy1", "meanest_fy2", "highest_fy1", "highest_fy2",
            "lowest_fy1", "lowest_fy2", "at", "lt", "dlyvol")

    def __init__(self, horizon: int = 5, span: int = 20):
        self.horizon = horizon
        self.span = span

    def transform(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, str, List[str]]:
        """Builds the confounded forward-return feature frame.

        Args:
            df: Base dataset with daily returns and Fama-French factors.

        Returns:
            Tuple of (processed frame, target column name, confounder list).

        Raises:
            ValueError: If the input frame is empty.
            KeyError: If required baseline columns are missing.
        """
        if df.empty:
            raise ValueError("Input DataFrame is empty.")
        required = ["dlyret", "mktrf", "smb", "hml"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise KeyError(f"Missing required columns: {missing}")

        out = df.copy()
        out["ar"] = out["dlyret"] - out["mktrf"]
        target = f"car_fwd_{self.horizon}d"
        shifts = [out["ar"].shift(-i) for i in range(1, self.horizon + 1)]
        out[target] = np.add.reduce(shifts)

        ewm = out["dlyret"].ewm(span=self.span, adjust=False)
        out["vol_ewm"] = ewm.std() * np.sqrt(252)
        out["mom_ewm"] = ewm.mean()

        confounders = list(self._BASE) + [c for c in self._OPT if c in out.columns]
        out = out.dropna(subset=[target] + confounders).reset_index(drop=True)
        return out, target, confounders

In [22]:
import numpy as np
import pandas as pd
from typing import List, Dict
from scipy.stats import norm
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MultiLabelBinarizer


class ContinuousCausalEstimator:
    """Vectorized Double Machine Learning for continuous ATE estimation.

    Attributes:
        y_col: Continuous outcome column.
        e_col: Event-array column.
        x_cols: Confounder column names.
        cv: Cross-fitting fold count.
        min_obs: Minimum treatments required per event to retain it.
        seed: Deterministic seed.
    """

    def __init__(
        self,
        y_col: str,
        e_col: str,
        x_cols: List[str],
        cv: int = 5,
        min_obs: int = 1,
        seed: int = 42,
    ):
        self.y_col = y_col
        self.e_col = e_col
        self.x_cols = x_cols
        self.cv = cv
        self.min_obs = min_obs
        self.seed = seed
        self._mlb = MultiLabelBinarizer()

    def _treatments(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extracts the binary treatment matrix from the encoded event column.

        Args:
            df: Source frame containing the event column.

        Returns:
            DataFrame of 0/1 treatment indicators, one column per qualifying event.
        """
        events = df[self.e_col].astype(str).str.findall(r"\d+").apply(lambda xs: [int(x) for x in xs])
        t = pd.DataFrame(self._mlb.fit_transform(events), columns=self._mlb.classes_, index=df.index)
        return t.loc[:, t.sum(axis=0) >= self.min_obs]

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Runs the orthogonalized partially-linear DML estimator.

        Args:
            df: Integrated feature matrix with the target and event columns.

        Returns:
            DataFrame of per-event ATE, standard error, t-statistic and p-value,
            sorted by absolute t-statistic.
        """
        d = df.dropna(subset=[self.y_col, self.e_col]).copy()
        x = d[self.x_cols].to_numpy(dtype=np.float64)
        y = d[self.y_col].to_numpy(dtype=np.float64)

        t_df = self._treatments(d)
        if t_df.empty:
            return pd.DataFrame()
        t = t_df.to_numpy(dtype=np.float64)

        kf = KFold(n_splits=self.cv, shuffle=True, random_state=self.seed)
        base = HistGradientBoostingRegressor(random_state=self.seed)
        multi = MultiOutputRegressor(base)

        t_res = t - cross_val_predict(multi, x, t, cv=kf, n_jobs=-1)
        y_res = y - cross_val_predict(base, x, y, cv=kf, n_jobs=-1)

        ss = np.einsum("ij,ij->j", t_res, t_res)
        keep = ss > 0
        t_res, ss = t_res[:, keep], ss[keep]
        ids = t_df.columns[keep]
        n_obs = t[:, keep].sum(axis=0).astype(int)

        ate = np.einsum("ij,i->j", t_res, y_res) / ss
        eps = y_res[:, None] - t_res * ate[None, :]
        var = np.einsum("ij,ij->j", t_res * eps, t_res * eps) / ss**2
        se = np.sqrt(var)
        tstat = ate / se

        return (
            pd.DataFrame({
                "event_id": ids,
                "n_obs": n_obs,
                "ate": ate,
                "se": se,
                "t_stat": tstat,
                "p_val": 2 * norm.sf(np.abs(tstat)),
            })
            .sort_values("t_stat", key=np.abs, ascending=False)
            .reset_index(drop=True)
        )

In [23]:
class ContinuousDistributionEstimator:
    """Gradient-boosted quantile regression for the conditional return CDF.

    Attributes:
        y_col: Continuous outcome column.
        e_col: Event-array column.
        x_cols: Confounder column names.
        quantiles: Quantile levels at which the CDF is evaluated.
        seed: Deterministic seed.
    """

    def __init__(
        self,
        y_col: str,
        e_col: str,
        x_cols: List[str],
        quantiles: List[float] | None = None,
        seed: int = 42,
    ):
        self.y_col = y_col
        self.e_col = e_col
        self.x_cols = x_cols
        self.quantiles = quantiles or [0.05, 0.25, 0.50, 0.75, 0.95]
        self.seed = seed
        self.models: Dict[float, HistGradientBoostingRegressor] = {}
        self._mlb = MultiLabelBinarizer()

    def _design(self, df: pd.DataFrame, fit: bool) -> np.ndarray:
        """Concatenates confounders with multi-label treatment indicators.

        Args:
            df: Frame containing confounders and events.
            fit: Whether to fit the binarizer or only transform.

        Returns:
            Float64 design matrix.
        """
        events = df[self.e_col].astype(str).str.findall(r"\d+").apply(lambda xs: [int(x) for x in xs])
        codes = self._mlb.fit_transform(events) if fit else self._mlb.transform(events)
        return np.hstack([df[self.x_cols].to_numpy(dtype=np.float64), codes.astype(np.float64)])

    def fit(self, df: pd.DataFrame) -> "ContinuousDistributionEstimator":
        """Fits one HGBT quantile regressor per target quantile.

        Args:
            df: Integrated feature matrix.

        Returns:
            Fitted instance.
        """
        d = df.dropna(subset=[self.y_col, self.e_col]).copy()
        x = self._design(d, fit=True)
        y = d[self.y_col].to_numpy(dtype=np.float64)
        for q in self.quantiles:
            m = HistGradientBoostingRegressor(loss="quantile", quantile=q, random_state=self.seed)
            m.fit(x, y)
            self.models[q] = m
        return self

    def predict_distribution(self, df: pd.DataFrame) -> pd.DataFrame:
        """Predicts the conditional return quantiles, enforcing monotonicity.

        Args:
            df: Feature matrix with target events to evaluate.

        Returns:
            DataFrame of predicted quantiles, monotonically sorted across columns.
        """
        x = self._design(df, fit=False)
        out = pd.DataFrame(index=df.index)
        for q, m in self.models.items():
            out[f"q_{q:.2f}"] = m.predict(x)
        out.iloc[:, :] = np.maximum.accumulate(out.to_numpy(), axis=1)
        return out

In [24]:
import numpy as np
import pandas as pd
from typing import List, Dict
from lifelines import WeibullAFTFitter


class SurvivalDataBuilder:
    """Builds right-censored time-to-event durations from event panel data."""

    @staticmethod
    def build(df: pd.DataFrame, target_id: int, event_col: str = "keydeveventtypeid") -> pd.DataFrame:
        """Computes time-to-next-event for each observation row.

        Args:
            df: Panel data with a datetime column and encoded event sequences.
            target_id: Numerical identifier for the terminal event.
            event_col: Column containing the encoded event sequence strings.

        Returns:
            DataFrame augmented with ``duration`` (days) and ``observed`` (bool).
        """
        d = df.copy()
        d["date"] = pd.to_datetime(d["dlycaldt"])
        mask = d[event_col].astype(str).str.contains(rf"\b{target_id}\b", regex=True)
        event_dates = d.loc[mask, "date"].dropna().sort_values().to_numpy()
        dates = d["date"].to_numpy()

        if event_dates.size == 0:
            d["duration"] = (dates.max() - dates).astype("timedelta64[D]").astype(float)
            d["observed"] = False
            return d[d["duration"] > 0].reset_index(drop=True)

        idx = np.searchsorted(event_dates, dates, side="right")
        censored = idx == event_dates.size
        nxt = np.empty_like(dates)
        nxt[~censored] = event_dates[idx[~censored]]
        nxt[censored] = dates.max()

        d["duration"] = (nxt - dates).astype("timedelta64[D]").astype(float)
        d["observed"] = ~censored
        return d[d["duration"] > 0].reset_index(drop=True)


class WeibullTimingEstimator:
    """Weibull AFT survival model for event timing forecasts.

    Attributes:
        features: Covariate columns passed to the regression.
        penalizer: L2 penalty on the log-likelihood.
        model: Underlying lifelines fitter instance.
    """

    def __init__(self, features: List[str], penalizer: float = 0.01):
        from lifelines import WeibullAFTFitter
        self.features = features
        self.penalizer = penalizer
        self.model = WeibullAFTFitter(penalizer=penalizer)

    def fit(self, df: pd.DataFrame) -> "WeibullTimingEstimator":
        """Fits the AFT model on the survival-augmented frame.

        Args:
            df: Frame containing features plus ``duration`` and ``observed``.

        Returns:
            Fitted instance.
        """
        fit_df = df[self.features + ["duration", "observed"]].dropna()
        self.model.fit(fit_df, duration_col="duration", event_col="observed", show_progress=False)
        return self

    def _survival(self, x: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """Extracts the fitted survival function as numpy arrays.

        Args:
            x: Single-row feature frame.

        Returns:
            Tuple of (time grid, survival probability grid).
        """
        sf = self.model.predict_survival_function(x[self.features])
        return sf.index.to_numpy(dtype=float), sf.iloc[:, 0].to_numpy(dtype=float)

    def predict_buckets(self, x: pd.DataFrame, buckets: List[int] | None = None) -> Dict[str, float]:
        """Aggregates survival mass into labelled time buckets.

        Args:
            x: Single-row feature frame.
            buckets: Bucket boundaries in days; defaults to [30, 60, 90].

        Returns:
            Dict mapping bucket label to percentage probability mass.
        """
        buckets = buckets or [30, 60, 90]
        idx, val = self._survival(x)
        s = np.interp(np.asarray(buckets, dtype=float), idx, val)
        probs = {f"0-{buckets[0]} Days": 1.0 - s[0]}
        for i in range(len(buckets) - 1):
            probs[f"{buckets[i]}-{buckets[i+1]} Days"] = s[i] - s[i + 1]
        probs[f"{buckets[-1]}+ Days"] = s[-1]
        return {k: round(v * 100, 2) for k, v in probs.items()}

    def predict_daily(self, x: pd.DataFrame, max_days: int = 252) -> pd.DataFrame:
        """Produces the daily survival, cumulative and marginal probability curves.

        Args:
            x: Single-row feature frame.
            max_days: Upper bound of the daily forecast horizon.

        Returns:
            DataFrame with columns day, survival_prob, cumulative_prob, marginal_prob.
        """
        idx, val = self._survival(x)
        days = np.arange(1, max_days + 1, dtype=float)
        s = np.interp(days, idx, val)
        lag = np.insert(s[:-1], 0, 1.0)
        return pd.DataFrame({
            "day": days.astype(int),
            "survival_prob": s,
            "cumulative_prob": 1.0 - s,
            "marginal_prob": lag - s,
        })

In [25]:
matrix_builder = FeatureMatrix(horizon=5, span=20)
df_features, target, confounders = matrix_builder.transform(df_main)

In [26]:
df_features

,dlycaldt,dlyret,dlyvol,curcd,at,lt,ni,revt,meanest_fy1,meanest_fy2,...,hml,rmw,cma,rf,keydeveventtypeid,headline,ar,car_fwd_5d,vol_ewm,mom_ewm
0,2022-06-06,0.033044,1651320.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,2.54,...,0.0066,0.0047,0.0017,0.0000,None,None,0.029744,0.046952,0.443645,-0.002715
1,2022-06-07,0.014731,2265246.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,2.54,...,-0.0015,-0.0157,0.0065,0.0000,None,None,0.004731,0.032848,0.344974,-0.001053
2,2022-06-08,0.037744,1899037.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,2.54,...,-0.0052,-0.0070,-0.0025,0.0000,None,None,0.047944,-0.038226,0.393561,0.002642
3,2022-06-09,-0.005196,1451446.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,2.54,...,0.0026,0.0085,0.0021,0.0000,None,None,0.019104,-0.009014,0.342604,0.001895
4,2022-06-10,-0.030936,1077745.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,2.54,...,0.0066,0.0029,0.0055,0.0000,None,None,-0.000936,-0.056774,0.362597,-0.001232
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887,2025-12-17,-0.012780,1897430.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,0.0132,0.0060,0.0099,0.0002,"[np.int64(16), np.int64(101)]",['Woodside Energy Announces Resignation of Meg...,-0.001180,-0.029676,0.194091,-0.005517
888,2025-12-18,-0.056311,2963872.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,-0.0083,-0.0035,-0.0042,0.0002,None,None,-0.064311,0.031344,0.305051,-0.010354
889,2025-12-19,0.021948,2402751.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,-0.0104,-0.0088,-0.0069,0.0002,None,None,0.012948,0.032913,0.328700,-0.007278
890,2025-12-22,0.022148,1339901.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,-0.0024,-0.0072,-0.0032,0.0002,None,None,0.015248,0.026108,0.342849,-0.004476


In [27]:
causal_estimator = ContinuousCausalEstimator(
    y_col=target,
    e_col='keydeveventtypeid',
    x_cols=confounders
)
ate_results = causal_estimator.fit_transform(df_features)

In [28]:
ate_results

,event_id,n_obs,ate,se,t_stat,p_val
0,52,2,-0.032664,0.009185,-3.556259,0.000376
1,101,1,-0.043872,0.016186,-2.710536,0.006717
2,77,3,0.042268,0.016664,2.536458,0.011198
3,3,2,-0.043076,0.021377,-2.015036,0.043901
4,12,2,-0.029757,0.016391,-1.815394,0.069463
5,28,7,0.019157,0.010652,1.798360,0.072120
6,226,20,0.012741,0.007316,1.741614,0.081576
7,194,6,-0.031991,0.018802,-1.701439,0.088861
8,55,9,0.023012,0.013548,1.698487,0.089416
9,42,1,-0.015244,0.009469,-1.609884,0.107423


In [29]:
dist_estimator = ContinuousDistributionEstimator(
    y_col=target,
    e_col='keydeveventtypeid',
    x_cols=confounders,
    quantiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)
dist_estimator.fit(df_features)

In [30]:
recent_events = df_features.dropna(subset=['keydeveventtypeid']).tail()
event_distributions = dist_estimator.predict_distribution(recent_events)

In [31]:
recent_events

,dlycaldt,dlyret,dlyvol,curcd,at,lt,ni,revt,meanest_fy1,meanest_fy2,...,hml,rmw,cma,rf,keydeveventtypeid,headline,ar,car_fwd_5d,vol_ewm,mom_ewm
847,2025-10-21,-0.003446,703292.0,USD,61264.0,25111.0,3573.0,13179.0,1.15,0.76,...,-0.0004,0.0050,0.0075,0.0002,"[np.int64(55), np.int64(149)]",['Woodside Energy Group Ltd to Report Fiscal Y...,-0.003346,0.074294,0.211444,-0.002945
848,2025-10-22,0.056017,1400126.0,USD,61264.0,25111.0,3573.0,13179.0,1.15,0.76,...,0.0047,0.0116,0.0008,0.0002,"[np.int64(226), np.int64(81), np.int64(81), np...",['Woodside Energy Group Ltd. Reports Productio...,0.062717,0.018778,0.346288,0.002670
873,2025-11-26,0.012308,498714.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,-0.0007,-0.0035,-0.0006,0.0002,[np.int64(149)],"['Reuters Events, Energy LIVE Conference 2025,...",0.005408,0.018581,0.230467,-0.000310
878,2025-12-04,0.000592,579776.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,-0.0069,-0.0065,-0.0058,0.0002,[np.int64(51)],['Woodside Energy Group Ltd Presents at Energy...,-0.000808,-0.036845,0.197521,0.002021
887,2025-12-17,-0.012780,1897430.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,0.0132,0.0060,0.0099,0.0002,"[np.int64(16), np.int64(101)]",['Woodside Energy Announces Resignation of Meg...,-0.001180,-0.029676,0.194091,-0.005517


In [32]:
event_distributions

,q_0.05,q_0.10,q_0.25,q_0.50,q_0.75,q_0.90,q_0.95
847,-0.063815,-0.044216,-0.027200,-0.002894,0.030285,0.074373,0.074373
848,-0.063346,-0.029444,-0.012807,0.018560,0.027132,0.041777,0.045519
873,-0.044644,-0.041740,-0.032710,0.004548,0.018069,0.018555,0.020937
878,-0.037113,-0.036923,-0.036841,-0.029489,-0.015618,0.014392,0.025001
887,-0.056917,-0.045001,-0.029649,-0.013839,0.007730,0.083971,0.083971


In [33]:
df_surv = SurvivalDataBuilder.build(df_features, target_id=83)

timing_est = WeibullTimingEstimator(features=confounders, penalizer=0.01)
timing_est.fit(df_surv)

state_t0 = df_surv.tail(1)

timing_dist = timing_est.predict_buckets(state_t0, buckets=[30, 60, 90])
df_timing = pd.DataFrame(list(timing_dist.items()), columns=['Time Horizon', 'Probability (%)'])

continuous_dist = timing_est.predict_daily(state_t0, max_days=90)
peak_hazards = continuous_dist.sort_values('marginal_prob', ascending=False).head(5)

In [34]:
continuous_dist

,day,survival_prob,cumulative_prob,marginal_prob
0,1,0.999991,0.000009,0.000009
1,2,0.999954,0.000046,0.000037
2,3,0.999880,0.000120,0.000074
3,4,0.999764,0.000236,0.000116
4,5,0.999600,0.000400,0.000164
...,...,...,...,...
85,86,0.719415,0.280585,0.006500
86,87,0.712915,0.287085,0.006500
87,88,0.706351,0.293649,0.006565
88,89,0.699746,0.300254,0.006605


In [35]:
peak_hazards

,day,survival_prob,cumulative_prob,marginal_prob
89,90,0.693102,0.306898,0.006643
88,89,0.699746,0.300254,0.006605
87,88,0.706351,0.293649,0.006565
85,86,0.719415,0.280585,0.006500
86,87,0.712915,0.287085,0.006500


In [36]:
df_timing.to_string(index=False)

'Time Horizon  Probability (%)\n   0-30 Days             2.71\n  30-60 Days            10.42\n  60-90 Days            17.56\n    90+ Days            69.31'

# Options

In [37]:
df_opt

,root,expiry,T,type,strike,price,iv,px_last,px_settle,ivol,ticker
0,2WDS,2026-04-09,0.005476,Call,28.5,3.565,0.745070,3.565,3.565,74.50698,2WDS AU 04/09/26 C28.5 Equity
1,2WDS,2026-04-09,0.005476,Call,29.0,3.065,NaN,3.065,3.065,NaN,2WDS AU 04/09/26 C29 Equity
2,2WDS,2026-04-09,0.005476,Call,29.5,2.565,0.542726,2.565,2.565,54.27263,2WDS AU 04/09/26 C29.5 Equity
3,2WDS,2026-04-09,0.005476,Call,30.0,2.070,NaN,2.070,2.070,NaN,2WDS AU 04/09/26 C30 Equity
4,2WDS,2026-04-09,0.005476,Call,30.5,1.595,0.555944,1.595,1.595,55.59438,2WDS AU 04/09/26 C30.5 Equity
...,...,...,...,...,...,...,...,...,...,...,...
1392,WDS,2028-12-21,2.707734,Put,36.0,8.215,0.321290,8.215,8.215,32.12903,WDS AU 12/21/28 P36 Equity
1393,WDS,2028-12-21,2.707734,Put,37.0,8.845,0.299948,8.845,8.845,29.99479,WDS AU 12/21/28 P37 Equity
1394,WDS,2028-12-21,2.707734,Put,38.0,9.465,0.298211,9.465,9.465,29.82107,WDS AU 12/21/28 P38 Equity
1395,WDS,2028-12-21,2.707734,Put,39.0,10.075,0.295053,10.075,10.075,29.50533,WDS AU 12/21/28 P39 Equity


In [38]:
@dataclass(slots=True)
class SABRParams:
    """Calibrated SABR parameters for a single expiry.

    Attributes:
        alpha: Initial volatility level.
        beta: CEV exponent held fixed in calibration.
        rho: Correlation between forward and vol processes.
        nu: Volatility of volatility.
        rmse: In-sample RMSE against market implied volatilities.
    """

    alpha: float
    beta: float
    rho: float
    nu: float
    rmse: float


class SABR:
    """Hagan 2002 SABR implied-volatility formula with Obłój 2008 stabilization.

    Provides the HKLW lognormal implied volatility and a fast 2D calibration
    that fixes beta and pins alpha to the ATM volatility by solving the
    induced cubic, reducing the optimization to (rho, nu).
    """

    @staticmethod
    def iv(f: float, k: np.ndarray, t: float, alpha: float, beta: float, rho: float, nu: float) -> np.ndarray:
        """Computes SABR lognormal implied volatility on a strike vector.

        Args:
            f: Forward price.
            k: Strike or array of strikes.
            t: Time to expiry in years.
            alpha: SABR alpha.
            beta: SABR beta.
            rho: SABR rho.
            nu: SABR nu.

        Returns:
            Array of lognormal implied volatilities matching the shape of ``k``.
        """
        k = np.asarray(k, dtype=float)
        eps = 1e-12
        one_b = 1.0 - beta
        fk = f * k
        log_fk = np.log(f / k)

        fk_pow = fk ** (one_b / 2.0)
        pre = alpha / (fk_pow * (1.0 + one_b**2 / 24.0 * log_fk**2 + one_b**4 / 1920.0 * log_fk**4))

        z = (nu / alpha) * fk_pow * log_fk
        sqrt_term = np.sqrt(1.0 - 2.0 * rho * z + z**2)
        x_z = np.log((sqrt_term + z - rho) / (1.0 - rho))
        ratio = np.where(np.abs(z) < 1e-07, 1.0, z / np.where(np.abs(x_z) < eps, eps, x_z))

        correction = 1.0 + (
            one_b**2 / 24.0 * alpha**2 / (fk_pow**2)
            + 0.25 * rho * beta * nu * alpha / fk_pow
            + (2.0 - 3.0 * rho**2) / 24.0 * nu**2
        ) * t

        return pre * ratio * correction

    @classmethod
    def calibrate(
        cls,
        f: float,
        k: np.ndarray,
        iv_mkt: np.ndarray,
        t: float,
        beta: float = 1.0,
        weights: np.ndarray | None = None,
    ) -> SABRParams:
        """Calibrates (rho, nu) with alpha pinned to the ATM volatility.

        Fixing beta and solving alpha as the positive real root of the cubic
        induced by the ATM condition reduces the calibration to a
        well-conditioned 2D minimization. Multi-start L-BFGS-B is used to
        avoid local minima.

        Args:
            f: Forward price.
            k: Strike vector of market quotes.
            iv_mkt: Corresponding market implied volatilities.
            t: Time to expiry in years.
            beta: CEV exponent held fixed during calibration.
            weights: Optional per-quote weights; defaults to uniform.

        Returns:
            Fitted SABRParams container including the in-sample RMSE.
        """
        k = np.asarray(k, dtype=float)
        iv_mkt = np.asarray(iv_mkt, dtype=float)
        w = np.ones_like(iv_mkt) if weights is None else np.asarray(weights, dtype=float)

        order = np.argsort(k)
        atm_iv = float(np.interp(f, k[order], iv_mkt[order]))

        def alpha_from_atm(rho: float, nu: float) -> float:
            """Solves the ATM cubic for alpha given (rho, nu, beta, f, t, atm_iv).

            Args:
                rho: Current SABR rho.
                nu: Current SABR nu.

            Returns:
                Smallest positive real root, or the leading-order guess on failure.
            """
            one_b = 1.0 - beta
            f_pow = f ** one_b
            c3 = one_b**2 * t / (24.0 * f_pow**3)
            c2 = 0.25 * rho * beta * nu * t / (f_pow**2)
            c1 = (1.0 + (2.0 - 3.0 * rho**2) / 24.0 * nu**2 * t) / f_pow
            c0 = -atm_iv
            roots = np.roots([c3, c2, c1, c0])
            real = roots[np.abs(roots.imag) < 1e-08].real
            pos = real[real > 0]
            return float(pos.min()) if pos.size else atm_iv * f_pow

        def loss(theta: np.ndarray) -> float:
            """Weighted RMSE between SABR and market IVs for given (rho, nu).

            Args:
                theta: Length-2 array [rho, nu].

            Returns:
                Weighted root-mean-squared error across quotes.
            """
            rho, nu = theta
            alpha = alpha_from_atm(rho, nu)
            model = cls.iv(f, k, t, alpha, beta, rho, nu)
            return float(np.sqrt(np.mean(w * (model - iv_mkt) ** 2)))

        best = None
        for rho0 in (-0.5, -0.2, 0.0, 0.2, 0.5):
            for nu0 in (0.2, 0.5, 1.0, 1.5):
                res = minimize(
                    loss,
                    x0=np.array([rho0, nu0]),
                    method="L-BFGS-B",
                    bounds=[(-0.999, 0.999), (1e-04, 5.0)],
                )
                if res.success and (best is None or res.fun < best.fun):
                    best = res

        rho, nu = best.x
        alpha = alpha_from_atm(rho, nu)
        return SABRParams(alpha=alpha, beta=beta, rho=rho, nu=nu, rmse=float(best.fun))


@dataclass(slots=True)
class BLResult:
    """Container for one expiry's Breeden-Litzenberger output.

    Attributes:
        expiry: Expiry date.
        T: Time to expiry in years.
        forward: Forward price inferred from put-call parity.
        discount: Discount factor.
        rate: Continuously compounded zero rate.
        sabr: Calibrated SABR parameters for this expiry.
        strike_grid: Dense strike grid for the density.
        iv_curve: SABR implied volatilities on the strike grid.
        call_curve: Black-76 call prices on the strike grid.
        density: Risk-neutral density on the strike grid.
        parity_rmse: RMSE of the put-call parity regression.
        arb_violations: Count of negative density entries prior to clipping.
        n_quotes: Quotes retained for this expiry.
    """

    expiry: pd.Timestamp
    T: float
    forward: float
    discount: float
    rate: float
    sabr: SABRParams
    strike_grid: np.ndarray
    iv_curve: np.ndarray
    call_curve: np.ndarray
    density: np.ndarray
    parity_rmse: float
    arb_violations: int
    n_quotes: int

    def moments(self) -> Dict[str, float]:
        """Computes risk-neutral mean, variance, skewness and excess kurtosis.

        Returns:
            Dict of moment estimates via trapezoidal integration.
        """
        k, q = self.strike_grid, self.density
        tot = np.trapezoid(q, k)
        if tot <= 0:
            return {"mean": np.nan, "variance": np.nan, "skew": np.nan, "kurt": np.nan}
        qn = q / tot
        m1 = float(np.trapezoid(k * qn, k))
        dev = k - m1
        m2 = float(np.trapezoid(dev**2 * qn, k))
        sd = np.sqrt(m2) if m2 > 0 else np.nan
        if np.isnan(sd):
            return {"mean": m1, "variance": m2, "skew": np.nan, "kurt": np.nan}
        m3 = float(np.trapezoid(dev**3 * qn, k) / sd**3)
        m4 = float(np.trapezoid(dev**4 * qn, k) / sd**4)
        return {"mean": m1, "variance": m2, "skew": m3, "kurt": m4}

In [39]:

class BreedenLitzenberger:
    """SABR-based risk-neutral density extraction from an option chain.

    For each expiry the pipeline infers the forward via put-call parity,
    synthesizes OTM call prices, inverts Black-76 implied volatilities,
    calibrates a SABR smile with beta fixed, then evaluates the second
    strike derivative of the SABR call surface to obtain the density.

    Attributes:
        roots: Root tickers treated as European.
        min_quotes: Minimum retained quotes required to fit an expiry.
        n_grid: Strike grid resolution.
        tail: Log-moneyness extrapolation on each side of the observed range.
        beta: SABR beta held fixed during calibration.
        min_t: Minimum time-to-expiry required to attempt calibration.
        failures: Populated by :meth:`fit` with per-expiry error messages.
    """

    def __init__(
        self,
        roots: set[str] | None = None,
        min_quotes: int = 10,
        n_grid: int = 400,
        tail: float = 0.5,
        beta: float = 1.0,
        min_t: float = 7.0 / 365.25,
    ):
        self.roots = roots or {"WDSE"}
        self.min_quotes = min_quotes
        self.n_grid = n_grid
        self.tail = tail
        self.beta = beta
        self.min_t = min_t
        self.failures: Dict[pd.Timestamp, str] = {}

    def fit(self, df_opt: pd.DataFrame) -> Dict[pd.Timestamp, BLResult]:
        """Runs the full BL-SABR pipeline over every eligible expiry.

        Args:
            df_opt: Long-format option chain.

        Returns:
            Dict mapping expiry timestamp to BLResult.

        Raises:
            ValueError: If no quotes match the configured roots.
        """
        df = df_opt[df_opt["root"].isin(self.roots)]
        if df.empty:
            raise ValueError(f"No quotes matched {self.roots}.")

        self.failures.clear()
        out: Dict[pd.Timestamp, BLResult] = {}
        for expiry, sub in df.groupby("expiry", sort=True):
            if len(sub) < self.min_quotes:
                self.failures[expiry] = f"Insufficient quotes ({len(sub)} < {self.min_quotes})"
                continue
            try:
                res = self._fit_one(expiry, sub)
                if res is None:
                    self.failures[expiry] = "Pipeline returned None"
                else:
                    out[expiry] = res
            except Exception as exc:
                self.failures[expiry] = f"{type(exc).__name__}: {exc}"
        return out

    def summary(self, results: Dict[pd.Timestamp, BLResult]) -> pd.DataFrame:
        """Flattens a results dict into a per-expiry diagnostic table.

        Args:
            results: Output of :meth:`fit`.

        Returns:
            DataFrame of expiry-level metrics sorted by expiry.
        """
        rows = []
        for expiry, r in results.items():
            m = r.moments()
            rows.append({
                "expiry": expiry,
                "T": r.T,
                "forward": r.forward,
                "rate": r.rate,
                "n_quotes": r.n_quotes,
                "parity_rmse": r.parity_rmse,
                "sabr_alpha": r.sabr.alpha,
                "sabr_rho": r.sabr.rho,
                "sabr_nu": r.sabr.nu,
                "sabr_rmse": r.sabr.rmse,
                "arb_violations": r.arb_violations,
                "rnd_mean": m["mean"],
                "rnd_stdev": np.sqrt(m["variance"]) if np.isfinite(m["variance"]) else np.nan,
                "rnd_skew": m["skew"],
                "rnd_kurt": m["kurt"],
            })
        if not rows:
            return pd.DataFrame(columns=["expiry", "T", "forward", "rate", "n_quotes", "parity_rmse", "sabr_alpha", "sabr_rho", "sabr_nu", "sabr_rmse", "arb_violations", "rnd_mean", "rnd_stdev", "rnd_skew", "rnd_kurt"])
        return pd.DataFrame(rows).sort_values("expiry").reset_index(drop=True)

    def _fit_one(self, expiry: pd.Timestamp, sub: pd.DataFrame) -> BLResult | None:
        """Fits a single expiry's SABR smile and derives the density.

        Args:
            expiry: Timestamp of the expiry.
            sub: Option quotes filtered to this expiry.

        Returns:
            BLResult on success, or None if any stage has insufficient data.
        """
        t = float(sub["T"].iloc[0])
        if t < self.min_t:
            return None

        f, disc, parity_rmse = self._forward(sub)
        if not (np.isfinite(f) and f > 0 and np.isfinite(disc) and disc > 0):
            return None
        r = -np.log(disc) / t

        strikes, calls = self._otm_calls(sub, f, disc)
        if strikes.size < self.min_quotes:
            return None

        iv = self._implied_vol(calls, strikes, f, t, disc)
        ok = np.isfinite(iv) & (iv > 0.02) & (iv < 3.0)
        strikes, iv = strikes[ok], iv[ok]
        strikes, idx = np.unique(strikes, return_index=True)
        iv = iv[idx]
        if strikes.size < self.min_quotes:
            return None

        params = SABR.calibrate(f, strikes, iv, t, beta=self.beta)

        log_m = np.log(strikes / f)
        k_grid = np.linspace(log_m.min() - self.tail, log_m.max() + self.tail, self.n_grid)
        K = f * np.exp(k_grid)

        iv_grid = SABR.iv(f, K, t, params.alpha, params.beta, params.rho, params.nu)
        iv_grid = np.clip(iv_grid, 0.02, 3.0)
        calls_grid = self._black76(f, K, t, iv_grid, disc)
        density = np.exp(r * t) * np.gradient(np.gradient(calls_grid, K), K)

        arb = int(np.sum(density < 0))
        density = np.clip(density, 0.0, None)
        total = np.trapezoid(density, K)
        if total > 0:
            density /= total

        return BLResult(
            expiry=expiry, T=t, forward=float(f), discount=float(disc), rate=float(r),
            sabr=params, strike_grid=K, iv_curve=iv_grid, call_curve=calls_grid,
            density=density, parity_rmse=parity_rmse, arb_violations=arb, n_quotes=int(strikes.size),
        )

    @staticmethod
    def _forward(sub: pd.DataFrame) -> Tuple[float, float, float]:
        """Regresses put-call parity to infer the forward and discount factor.

        Args:
            sub: Option quotes for a single expiry.

        Returns:
            Tuple of (forward, discount, regression RMSE).
        """
        calls = sub[sub["type"] == "Call"].drop_duplicates("strike", keep="last")
        puts = sub[sub["type"] == "Put"].drop_duplicates("strike", keep="last")
        merged = calls[["strike", "price"]].merge(puts[["strike", "price"]], on="strike", suffixes=("_c", "_p"))
        if len(merged) < 3:
            return np.nan, np.nan, np.nan

        k = merged["strike"].to_numpy(float)
        y = (merged["price_c"] - merged["price_p"]).to_numpy(float)
        a = np.column_stack([np.ones_like(k), k])
        beta, *_ = np.linalg.lstsq(a, y, rcond=None)
        disc = -beta[1]
        if disc <= 0:
            return np.nan, np.nan, np.nan
        f = beta[0] / disc
        rmse = float(np.sqrt(np.mean((y - a @ beta) ** 2)))
        return float(f), float(disc), rmse

    @staticmethod
    def _otm_calls(sub: pd.DataFrame, f: float, disc: float) -> Tuple[np.ndarray, np.ndarray]:
        """Builds an OTM call curve, converting OTM puts via put-call parity.

        Args:
            sub: Option quotes for a single expiry.
            f: Forward price.
            disc: Discount factor.

        Returns:
            Tuple of (sorted strikes, synthetic call prices).
        """
        calls = sub[sub["type"] == "Call"]
        puts = sub[sub["type"] == "Put"]
        kc, vc = calls["strike"].to_numpy(float), calls["price"].to_numpy(float)
        kp, vp = puts["strike"].to_numpy(float), puts["price"].to_numpy(float)

        c_mask = kc >= f
        p_mask = kp < f
        kp_o, vp_o = kp[p_mask], vp[p_mask]
        synthetic = vp_o + disc * (f - kp_o)

        k = np.concatenate([kp_o, kc[c_mask]])
        v = np.concatenate([synthetic, vc[c_mask]])
        valid = v > 0
        k, v = k[valid], v[valid]
        order = np.argsort(k)
        return k[order], v[order]

    @staticmethod
    def _black76(
        f: float | np.ndarray,
        k: float | np.ndarray,
        t: float,
        sigma: float | np.ndarray,
        disc: float,
    ) -> np.ndarray:
        """Black-76 call pricing, vectorized via scipy.special.ndtr.

        Args:
            f: Forward price(s).
            k: Strike(s).
            t: Time to expiry in years.
            sigma: Lognormal volatility value(s).
            disc: Discount factor.

        Returns:
            Call prices broadcast to the largest input shape.
        """
        f = np.asarray(f, float)
        k = np.asarray(k, float)
        sigma = np.asarray(sigma, float)
        vol = np.maximum(sigma * np.sqrt(t), 1e-12)
        d1 = (np.log(f / k) + 0.5 * vol**2) / vol
        return disc * (f * ndtr(d1) - k * ndtr(d1 - vol))

    @classmethod
    def _implied_vol(
        cls,
        price: np.ndarray,
        k: np.ndarray,
        f: float,
        t: float,
        disc: float,
        tol: float = 1e-06,
        max_iter: int = 60,
    ) -> np.ndarray:
        """Vectorized bisection inversion of Black-76 for implied volatility.

        Args:
            price: Market call prices.
            k: Corresponding strikes.
            f: Forward price.
            t: Time to expiry in years.
            disc: Discount factor.
            tol: Absolute tolerance on the bracket width.
            max_iter: Maximum bisection iterations.

        Returns:
            Array of implied vols; NaN where no-arbitrage bounds are violated.
        """
        price = np.asarray(price, float)
        k = np.asarray(k, float)
        lb = disc * np.maximum(f - k, 0.0)
        ub = disc * f
        feasible = (price > lb - 1e-10) & (price < ub + 1e-10)

        lo = np.full_like(price, 1e-04)
        hi = np.full_like(price, 5.0)
        for _ in range(max_iter):
            mid = 0.5 * (lo + hi)
            low = cls._black76(f, k, t, mid, disc) < price
            lo = np.where(low, mid, lo)
            hi = np.where(low, hi, mid)
            if np.max(hi - lo) < tol:
                break
        return np.where(feasible, 0.5 * (lo + hi), np.nan)

In [40]:
all_roots = set(df_opt["root"].unique())
bl = BreedenLitzenberger(
    roots=all_roots,
    tail=0.5,
    min_quotes=5
)
results = bl.fit(df_opt)
summary = bl.summary(results)

In [42]:
results

{Timestamp('2026-04-16 00:00:00'): BLResult(expiry=Timestamp('2026-04-16 00:00:00'), T=0.024640657084188913, forward=32.1052085773575, discount=0.9960657065870484, rate=0.1599816551604822, sabr=SABRParams(alpha=0.43856412635380465, beta=1.0, rho=np.float64(0.30037871677630096), nu=np.float64(4.141337378885912), rmse=0.022132799118989677), strike_grid=array([14.86000116, 14.9156142 , 14.97143536, 15.02746544, 15.0837052 ,
        15.14015544, 15.19681694, 15.2536905 , 15.3107769 , 15.36807695,
        15.42559144, 15.48332118, 15.54126697, 15.59942962, 15.65780994,
        15.71640874, 15.77522685, 15.83426509, 15.89352427, 15.95300523,
        16.0127088 , 16.0726358 , 16.13278708, 16.19316347, 16.25376582,
        16.31459497, 16.37565177, 16.43693708, 16.49845174, 16.56019662,
        16.62217258, 16.68438048, 16.74682119, 16.80949558, 16.87240453,
        16.93554891, 16.99892961, 17.06254751, 17.1264035 , 17.19049847,
        17.25483331, 17.31940892, 17.3842262 , 17.44928606, 17.5

In [41]:
summary

,expiry,T,forward,rate,n_quotes,parity_rmse,sabr_alpha,sabr_rho,sabr_nu,sabr_rmse,arb_violations,rnd_mean,rnd_stdev,rnd_skew,rnd_kurt
0,2026-04-16,0.024641,32.105209,0.159982,58,0.022122,0.438564,0.300379,4.141337,0.022133,0,32.103836,2.537891,1.352964,10.098885
1,2026-04-23,0.043806,32.165556,-0.093133,34,0.033982,0.421688,0.169867,2.180890,0.006162,0,32.165886,3.034270,0.689682,5.264798
2,2026-05-21,0.120465,32.216915,0.052697,70,0.024521,0.388256,0.127915,1.257515,0.010478,0,32.211715,4.615118,0.734972,4.950358
3,2026-06-18,0.197125,32.296449,0.036275,87,0.035874,0.363216,0.107354,0.916479,0.010152,0,32.289917,5.506747,0.772651,4.895845
4,2026-07-16,0.273785,32.372629,0.015981,47,0.041225,0.351213,0.020020,0.727668,0.005484,0,32.350354,6.181263,0.641517,4.172973
5,2026-08-20,0.369610,32.441536,0.013153,33,0.022700,0.334326,0.045062,0.715982,0.002125,0,32.389864,6.825006,0.705526,4.192680
6,2026-09-17,0.446270,32.053887,-0.021010,77,0.149932,0.342251,-0.212904,0.720141,0.008659,0,32.015870,7.553331,0.452789,3.866493
7,2026-12-17,0.695414,32.251080,0.009211,72,0.086612,0.305350,-0.004486,0.875592,0.007406,0,31.860514,8.272970,0.554898,4.059199
8,2027-03-18,0.944559,32.003076,0.007318,44,0.202312,0.307044,-0.141550,0.567392,0.008637,0,31.682168,9.306047,0.493422,3.408934
9,2027-06-17,1.193703,32.133011,-0.002061,26,0.080930,0.306559,-0.999000,0.049599,0.002516,0,31.823162,10.266076,0.588559,3.079952
